In [2]:
!pip install nltk spacy contractions gensim
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [25]:
import pandas as pd
import re
import nltk
import contractions

from nltk.corpus import brown, stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from gensim.models import Word2Vec
import gensim.downloader as api

nltk.download("brown")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [26]:
text = " ".join(" ".join(sentence) for sentence in brown.sents())

sentences = sent_tokenize(text)[:5000]

df = pd.DataFrame(sentences, columns=["text"])

print("Number of sentences:", len(df))
print(df["text"].head(10))

Number of sentences: 5000
0    The Fulton County Grand Jury said Friday an in...
1    The jury further said in term-end presentments...
2    The September-October term jury had been charg...
3    `` Only a relative handful of such reports was...
4    The jury said it did find that many of Georgia...
5    It recommended that Fulton legislators act `` ...
6    The grand jury commented on a number of other ...
7    Merger proposed However , the jury said it bel...
8    The City Purchasing Department , the jury said...
9    It urged that the city `` take steps to remedy...
Name: text, dtype: object


In [28]:
def clean_text(text):
    text = text.lower()
    text = contractions.fix(text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

print(df["clean_text"].head(10))

0    the fulton county grand jury said friday an in...
1    the jury further said in termend presentments ...
2    the septemberoctober term jury had been charge...
3    only a relative handful of such reports was re...
4    the jury said it did find that many of georgia...
5    it recommended that fulton legislators act to ...
6    the grand jury commented on a number of other ...
7    merger proposed however the jury said it belie...
8    the city purchasing department the jury said i...
9    it urged that the city take steps to remedy th...
Name: clean_text, dtype: object


In [29]:
stop_words = set(stopwords.words("english")) - {"not", "no", "nor"}

def remove_stopwords(text):
    words = text.split()
    return " ".join(word for word in words if word not in stop_words)

df["without_stopwords"] = df["clean_text"].apply(remove_stopwords)

print(df["without_stopwords"].head(10))

0    fulton county grand jury said friday investiga...
1    jury said termend presentments city executive ...
2    septemberoctober term jury charged fulton supe...
3    relative handful reports received jury said co...
4    jury said find many georgias registration elec...
5    recommended fulton legislators act laws studie...
6    grand jury commented number topics among atlan...
7    merger proposed however jury said believes two...
8    city purchasing department jury said lacking e...
9                 urged city take steps remedy problem
Name: without_stopwords, dtype: object


In [30]:
df["tokens"] = df["without_stopwords"].apply(word_tokenize)

print(df["tokens"].head(5))

0    [fulton, county, grand, jury, said, friday, in...
1    [jury, said, termend, presentments, city, exec...
2    [septemberoctober, term, jury, charged, fulton...
3    [relative, handful, reports, received, jury, s...
4    [jury, said, find, many, georgias, registratio...
Name: tokens, dtype: object


In [31]:
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df["lemmatized"] = df["tokens"].apply(lemmatize_tokens)

print(df["lemmatized"].head(5))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


0    [fulton, county, grand, jury, said, friday, in...
1    [jury, said, termend, presentment, city, execu...
2    [septemberoctober, term, jury, charged, fulton...
3    [relative, handful, report, received, jury, sa...
4    [jury, said, find, many, georgia, registration...
Name: lemmatized, dtype: object


In [32]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def stem_tokens(tokens):
    return [stemmer.stem(word) for word in tokens]

df["stemmed"] = df["tokens"].apply(stem_tokens)

print(df["stemmed"].head(5))

0    [fulton, counti, grand, juri, said, friday, in...
1    [juri, said, termend, present, citi, execut, c...
2    [septemberoctob, term, juri, charg, fulton, su...
3    [rel, hand, report, receiv, juri, said, consid...
4    [juri, said, find, mani, georgia, registr, ele...
Name: stemmed, dtype: object


In [33]:
from gensim.models import Word2Vec

word2vec_model = Word2Vec(
    sentences=df["lemmatized"].tolist(),
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

print("Word2Vec model trained successfully!")

Word2Vec model trained successfully!


In [34]:
def get_word2vec(tokens):
    vectors = [
        word2vec_model.wv[word]
        for word in tokens
        if word in word2vec_model.wv
    ]

    if not vectors:
        return [0] * 100

    return sum(vectors) / len(vectors)

df["word2vec"] = df["lemmatized"].apply(get_word2vec)

print(df["word2vec"].head())

0    [-0.028972939, 0.03378891, 0.011968753, 0.0201...
1    [-0.027024867, 0.031933155, 0.010478195, 0.020...
2    [-0.012776594, 0.014667529, 0.004201674, 0.007...
3    [-0.028579136, 0.033369202, 0.008527597, 0.019...
4    [-0.030017624, 0.03359499, 0.008889605, 0.0178...
Name: word2vec, dtype: object


In [35]:
import gensim.downloader as api

glove_model = api.load("glove-wiki-gigaword-50")

print("GloVe model loaded successfully!")

[==================================================] 100.0% 66.0/66.0MB downloaded
GloVe model loaded successfully!


In [36]:
def get_glove(tokens):
    vectors = [
        glove_model[word]
        for word in tokens
        if word in glove_model
    ]

    if not vectors:
        return [0] * 50

    return sum(vectors) / len(vectors)

df["glove"] = df["lemmatized"].apply(get_glove)

print(df["glove"].head())

0    [0.1257509, 0.13710952, -0.14327796, 0.5079608...
1    [-0.06951348, 0.10424352, -0.19383642, 0.24679...
2    [-0.011677816, -0.0966871, 0.031055948, 0.3427...
3    [0.11309822, 0.12045, 0.2254635, 0.014774365, ...
4    [-0.01380083, -0.2671162, -0.070279524, -0.053...
Name: glove, dtype: object
